# DG-TCAV 3D MRI Classifier — Clean Colab Launcher
Includes all supervisor directives and architectural upgrades:
- `configs/resnet18_medicalnet_regularized.yaml`: MedicalNet ResNet-18 + `dropout: 0.5` + L2 regularization + Medial-Temporal ROI Spatial Attention Pooling + 5-epoch head warm-up + differential LR + companion mask + ordinal continuum loss.
- `configs/hybrid_inception_resnet3d.yaml`: **Hybrid MedicalNet Stem (`conv1`..`layer3`) + 3D Inception-ResNet (`layer4`)** + `dropout: 0.5` + L2 regularization + ROI Attention Pooling.
- `configs/inception3d.yaml`: Low-compute pure 3D Inception (~3.1M params, 10x smaller).
- `configs/resnet18_medicalnet.yaml`: Locked Candidate C reference (`0.4459` Val Macro-F1, `48.31%` Val Acc).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone -b Inception-ResNet https://github.com/Alishals28/dg-tcav-classifier-fyp.git
%cd dg-tcav-classifier-fyp
!pip install -r requirements.txt
!pip install -e . --no-deps

In [ ]:
# Copy preprocessed 128x128x128 scans and companion masks to local Colab SSD for fast I/O
import glob, os, shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

src_dir = '/content/drive/MyDrive/DG-TCAV_FYP_NUST/DG_TCAV_OUTPUTS/preprocessed_128'
dst_dir = '/content/preprocessed_128'
os.makedirs(dst_dir, exist_ok=True)
files = glob.glob(os.path.join(src_dir, '*.nii.gz'))
print(f'Discovered {len(files)} NIfTI files (scans + masks) in Drive...')

def copy_worker(src_path):
    dest_path = os.path.join(dst_dir, os.path.basename(src_path))
    if not os.path.exists(dest_path) or os.path.getsize(dest_path) != os.path.getsize(src_path):
        shutil.copy2(src_path, dest_path)

with ThreadPoolExecutor(max_workers=32) as ex:
    list(tqdm(ex.map(copy_worker, files), total=len(files)))

In [ ]:
# Option A: Train Regularized MedicalNet ResNet-18 or Hybrid MedicalNet Inception-ResNet
CONFIG = 'configs/hybrid_inception_resnet3d.yaml'  # or 'configs/resnet18_medicalnet_regularized.yaml'
!python -m scripts.validate_dataset --config $CONFIG
!python -m src.train --config $CONFIG

In [ ]:
# Option B: Run the Supervisor's L2 Regularization (lambda) Sweep
# !python -m scripts.run_l2_sweep --config configs/resnet18_medicalnet_regularized.yaml --lambdas 0.001 0.005 0.01 0.05 --dropout 0.5